# Diagnóstico de Qualidade de Dados: CAPAG dos Municípios Brasileiros — Ano-Base 2025

**Objetivo:** Este notebook realiza uma verificação objetiva da base CAPAG 2025, com foco na qualidade dos dados e na preparação da base para integração com outras fontes do projeto.

A CAPAG será utilizada para identificar a situação fiscal dos municípios. No projeto, os municípios classificados como **C ou D** serão sinalizados como integrantes do critério de necessidade financeira.

---

## O que será verificado

Neste notebook serão realizadas as seguintes etapas:

1. Carregamento da base CAPAG 2025.
2. Seleção apenas das colunas relevantes para o projeto.
3. Verificação de valores nulos.
4. Verificação de registros duplicados.
5. Validação do código IBGE com 7 dígitos.
6. Análise da distribuição das classificações CAPAG.
7. Identificação dos municípios classificados como C ou D.
8. Padronização dos nomes das colunas.
9. Criação da variável `criterio_capag_cd`.
10. Exportação da base tratada para integração com SINISA e Atlas.

---

## Papel da CAPAG na análise

A CAPAG não será utilizada isoladamente para definir os municípios prioritários.

Ela será um dos critérios da análise:

- **CAPAG:** situação fiscal do município;
- **SINISA:** risco e informações sobre mapeamento;
- **Atlas Digital de Desastres:** histórico de ocorrências e impactos.

A integração entre essas bases será realizada posteriormente por meio do código IBGE.



## 1. Carregamento da CAPAG 2025

Carregar a base CAPAG referente ao ano-base 2025 para análise da situação fiscal dos municípios brasileiros.

In [45]:
import pandas as pd

planilha_id = "12VwMagHtwm_owaG0vobp3j5TK6lC6u9x"
url_export = f"https://docs.google.com/spreadsheets/d/{planilha_id}/export?format=xlsx"

df = pd.read_excel(
    url_export,
    sheet_name="CAPAG Ano Base 2025",
    header=3
)

print("Dimensão da base:", df.shape)
df.head()

Dimensão da base: (5567, 70)


/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,Código Município Completo,Nome_Município,UF,2025 - Dívida Consolidada,2025 - Receita Corrente Líquida,Indicador 1,Nota 1,2023 - Despesas Empenhadas,2023 - Receitas Correntes - Receitas Brutas Realizadas,2023 - Receitas Correntes - Deduções FUNDEB,...,DCB zerada ou negativa,Publicou RGF,OF negativa,Indicador 3,Insuficiência de caixa,Indicador 3 Antigo,Nota 3,CAPAG antes do ranking,Ranking da CCONF,CAPAG
0,5200050,Abadia de Goiás,GO,679436.95,1.139670e+08,0.005962,A,6.712108e+07,7.558667e+07,-4314496.84,...,NaN,Sim,NaN,0.103345,0.00,0.000448,A,A,Cicf,A
1,3100104,Abadia dos Dourados,MG,4263740.15,4.620732e+07,0.092274,A,3.372663e+07,4.003463e+07,-4915869.26,...,NaN,Sim,NaN,0.024377,-5740.80,0.298032,B,B,Dicf,B
2,5200100,Abadiânia,GO,14481365.03,9.427693e+07,0.153605,A,8.001550e+07,8.317818e+07,-7424954.17,...,NaN,Sim,NaN,-0.018015,-164204.73,3.110536,C,C,Cicf,C
3,3100203,Abaeté,MG,41600768.20,1.147456e+08,0.362548,A,8.707676e+07,9.658018e+07,-9468925.16,...,NaN,Sim,NaN,0.008619,-95491.10,0.127688,B,C,Aicf,C
4,1500107,Abaetetuba,PA,180.00,6.306541e+08,0.0,A,4.193456e+08,5.250544e+08,-30067247.36,...,NaN,Sim,NaN,-0.071245,-35213597.01,NaN,C,C,Bicf,C


## 2. Seleção das colunas realmente necessárias

Manter apenas as colunas necessárias para identificar o município, sua classificação fiscal e permitir o cruzamento posterior com outras bases do projeto.

In [46]:
colunas_capag = [
    "Código Município Completo",
    "Nome_Município",
    "UF",
    "CAPAG antes do ranking",
    "Ranking da CCONF",
    "CAPAG"
]

df_capag = df[colunas_capag].copy()

print("Dimensão da base selecionada:", df_capag.shape)
df_capag.head()

Dimensão da base selecionada: (5567, 6)


,Código Município Completo,Nome_Município,UF,CAPAG antes do ranking,Ranking da CCONF,CAPAG
0,5200050,Abadia de Goiás,GO,A,Cicf,A
1,3100104,Abadia dos Dourados,MG,B,Dicf,B
2,5200100,Abadiânia,GO,C,Cicf,C
3,3100203,Abaeté,MG,C,Aicf,C
4,1500107,Abaetetuba,PA,C,Bicf,C


## 3. Validação de nulos e duplicidades

Verificar a existência de valores nulos e registros duplicados nas variáveis selecionadas.

Também será conferida a unicidade do código IBGE, pois ele será utilizado posteriormente no cruzamento com outras bases do projeto.


In [47]:
print("Valores nulos por coluna:")
print(df_capag.isna().sum())

print("\nLinhas totalmente duplicadas:")
print(df_capag.duplicated().sum())

print("\nCódigos IBGE duplicados:")
print(
    df_capag["Código Município Completo"]
    .duplicated()
    .sum()
)

Valores nulos por coluna:
Código Município Completo    0
Nome_Município               0
UF                           0
CAPAG antes do ranking       0
Ranking da CCONF             0
CAPAG                        0
dtype: int64

Linhas totalmente duplicadas:
0

Códigos IBGE duplicados:
0


## 4. Validação do Código IBGE

Padronizar e validar o código IBGE dos municípios, garantindo que todos estejam no formato de 7 dígitos.

Esse campo será utilizado como chave de integração com outras bases do projeto.

In [48]:
df_capag["Código Município Completo"] = (
    df_capag["Código Município Completo"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(7)
)

print(
    "Códigos com 7 dígitos:",
    df_capag["Código Município Completo"].str.len().eq(7).sum()
)

print(
    "Total de registros:",
    len(df_capag)
)

print(
    "Códigos fora do padrão:",
    (~df_capag["Código Município Completo"].str.len().eq(7)).sum()
)

Códigos com 7 dígitos: 5567
Total de registros: 5567
Códigos fora do padrão: 0


## 5.Distribuição da classificação CAPAG

Analisar a distribuição das classificações CAPAG dos municípios brasileiros, com foco nas categorias C e D, que serão utilizadas como sinal de necessidade financeira no projeto.

In [49]:
distribuicao_capag = (
    df_capag["CAPAG"]
    .value_counts(dropna=False)
    .sort_index()
)

print(distribuicao_capag)

CAPAG
A       1190
A+       265
B        976
B+       113
C       2184
D         19
n.d.     683
n.e.     137
Name: count, dtype: int64


## 6. Seleção dos municípios C e D

Selecionar os municípios classificados como C ou D, que serão utilizados como grupo de interesse para o critério de necessidade financeira no cruzamento com as demais bases do projeto.

In [50]:
df_capag_cd = df_capag[
    df_capag["CAPAG"].isin(["C", "D"])
].copy()

print("Municípios com CAPAG C ou D:", len(df_capag_cd))

print("\nDistribuição:")
print(df_capag_cd["CAPAG"].value_counts())

Municípios com CAPAG C ou D: 2203

Distribuição:
CAPAG
C    2184
D      19
Name: count, dtype: int64


## 7. Padronização e criação do critério C/D

Padronizar os nomes das colunas e criar uma variável indicadora para identificar municípios classificados como CAPAG C ou D.

A base nacional completa será preservada para permitir o cruzamento posterior com SINISA e Atlas.

In [51]:
df_capag_final = df_capag.rename(columns={
    "Código Município Completo": "cod_ibge",
    "Nome_Município": "municipio",
    "UF": "uf",
    "CAPAG antes do ranking": "capag_pre_ranking",
    "Ranking da CCONF": "ranking_cconf",
    "CAPAG": "capag"
}).copy()

df_capag_final["criterio_capag_cd"] = (
    df_capag_final["capag"]
    .isin(["C", "D"])
)

print("Dimensão final:", df_capag_final.shape)

print("\nCritério CAPAG C/D:")
print(df_capag_final["criterio_capag_cd"].value_counts())

df_capag_final.head()

Dimensão final: (5567, 7)

Critério CAPAG C/D:
criterio_capag_cd
False    3364
True     2203
Name: count, dtype: int64


,cod_ibge,municipio,uf,capag_pre_ranking,ranking_cconf,capag,criterio_capag_cd
0,5200050,Abadia de Goiás,GO,A,Cicf,A,False
1,3100104,Abadia dos Dourados,MG,B,Dicf,B,False
2,5200100,Abadiânia,GO,C,Cicf,C,True
3,3100203,Abaeté,MG,C,Aicf,C,True
4,1500107,Abaetetuba,PA,C,Bicf,C,True


## 8. Exportação da base nacional tratada
Exportar a base nacional da CAPAG com os campos padronizados e o indicador que identifica municípios classificados como C ou D.

O arquivo será utilizado nas etapas seguintes de integração com SINISA e Atlas.

In [52]:
caminho_saida = "/content/drive/MyDrive/Colab Notebooks/capag_nacional_tratada.csv"

df_capag_final.to_csv(
    caminho_saida,
    index=False,
    encoding="utf-8-sig"
)

print("Arquivo exportado com sucesso!")
print("Caminho:", caminho_saida)
print("Dimensão:", df_capag_final.shape)

Arquivo exportado com sucesso!
Caminho: /content/drive/MyDrive/Colab Notebooks/capag_nacional_tratada.csv
Dimensão: (5567, 7)
